In [2]:
import sqlite3
import pandas as pd
# Database path - update this if your database is in a different location
DB_PATH = 'meme_evaluations.db'

# Connect to the database
def get_db_connection():
    """Get a connection to the SQLite database"""
    return sqlite3.connect(DB_PATH)

# 1. Session Summary
# =================
# Let's look at a summary of all evaluation sessions

def get_sessions():
    """Get all sessions from the database"""
    conn = get_db_connection()
    sessions_df = pd.read_sql_query("SELECT * FROM sessions ORDER BY timestamp DESC", conn)
    conn.close()
    return sessions_df

df = get_sessions()
df.head()

,id,name,timestamp,model
0,3,Baseline,2025-04-28T14:08:02.275375,gemini-1.5-flash
1,2,Baseline,2025-03-22T06:39:32.821437,gemini-1.5-flash
2,1,Baseline,2025-03-22T06:37:54.524915,gemini-1.5-flash


In [ ]:
def get_session_summary(session_id):
    """Get summary statistics for a specific session"""
    conn = get_db_connection()
    
    # Get session info
    session_info = pd.read_sql_query(
        "SELECT * FROM sessions WHERE id = ?", 
        conn, 
        params=[session_id]
    )
    
    # Get evaluation metrics
    metrics = pd.read_sql_query("""
        SELECT 
            COUNT(*) as total_memes,
            SUM(is_humor) as humor_count,
            SUM(is_creative) as creative_count,
            SUM(is_shareable) as shareable_count,
            COUNT(DISTINCT meme_template_id) as unique_templates
        FROM evaluations 
        WHERE session_id = ?
    """, conn, params=[session_id])
    
    conn.close()
    
    if len(session_info) == 0:
        print(f"No session found with ID {session_id}")
        return None, None
    
    return session_info, metrics

# baseline
session_info, metrics = get_session_summary('3')
session_info

,id,name,timestamp,model
0,3,Baseline,2025-04-28T14:08:02.275375,gemini-1.5-flash


In [4]:
metrics

,total_memes,humor_count,creative_count,shareable_count,unique_templates
0,29,6,3,12,6


In [5]:
conn = get_db_connection()
pd.read_sql_query("select * from evaluations where session_id=3", conn)

,id,session_id,prompt,meme_template_id,gemini_json,imgflip_url,is_humor,is_creative,is_shareable,comments,error,timestamp
0,30,3,"I am trying to make a funny meme, but my sense...",181913649,"{""template_id"": ""181913649"", ""text0"": ""Trying ...",https://i.imgflip.com/9s648m.jpg,0,0,1,Variation 1 selected. 2 out of 3 were drake,None,2025-04-28T14:08:42.849136
1,31,3,"I am trying to make a funny meme using AI, but...",129242436,"{""template_id"": ""129242436"", ""text0"": ""AI will...",https://i.imgflip.com/9s64a9.jpg,0,1,1,Variation 2 selected.,None,2025-04-28T14:09:36.252829
2,32,3,Pharmacy overlord,129242436,"{""template_id"": ""129242436"", ""text0"": ""Pharmac...",https://i.imgflip.com/9s64cf.jpg,0,0,0,Variation 1 selected. One option was very strange,None,2025-04-28T14:10:07.529183
3,33,3,Deep seek a Chinese AI model after being asked...,181913649,"{""template_id"": ""181913649"", ""text0"": ""Asking ...",https://i.imgflip.com/9s64dm.jpg,1,0,0,Variation 1 selected. Again drake,None,2025-04-28T14:10:49.336486
4,34,3,Yoda,181913649,"{""template_id"": ""181913649"", ""text0"": ""Using t...",https://i.imgflip.com/9s64fe.jpg,0,1,0,Variation 2 selected.,None,2025-04-28T14:12:22.660113
5,35,3,Prompt engineering vs fine-tuning,112126428,"{""template_id"": ""112126428"", ""boxes[0][text]"":...",https://i.imgflip.com/9s64jp.jpg,0,0,0,Variation 3 selected.,None,2025-04-28T14:12:53.934721
6,36,3,Please create me a funny meme about parents an...,217743513,"{""template_id"": ""217743513"", ""text0"": ""Treat t...",https://i.imgflip.com/9s64kv.jpg,0,0,0,Variation 2 selected. Still too much drake,None,2025-04-28T14:13:22.104129
7,37,3,Overengineering LLM Apps,181913649,"{""template_id"": ""181913649"", ""text0"": ""Writing...",https://i.imgflip.com/9s64m3.jpg,0,0,0,Variation 3 selected.,None,2025-04-28T14:13:47.014504
8,38,3,"I am trying to make a funny meme, but my sense...",217743513,"{""template_id"": ""217743513"", ""text0"": ""Make a ...",https://i.imgflip.com/9s64n0.jpg,0,0,0,Variation 2 selected.,None,2025-04-28T14:14:45.171367
9,39,3,i want to represent that a builder of an app w...,112126428,"{""template_id"": ""112126428"", ""boxes[0][text]"":...",https://i.imgflip.com/9s64pl.jpg,0,0,0,Variation 2 selected. Some problems here,None,2025-04-28T14:15:21.884412


In [6]:
# more context
# generally more shareable but also more errors 
session_info, metrics = get_session_summary('4')
display(session_info,metrics)

,id,name,timestamp,model
0,4,More memes,2025-04-28T14:39:33.040205,gemini-1.5-flash


,total_memes,humor_count,creative_count,shareable_count,unique_templates
0,29,6,0,20,12


In [7]:
# better model
# nothing but errors
session_info, metrics = get_session_summary('5')
display(session_info,metrics)

,id,name,timestamp,model
0,5,New model gemini 2 flash and more context,2025-04-28T15:07:43.387693,gemini-2.0-flash-lite


,total_memes,humor_count,creative_count,shareable_count,unique_templates
0,3,0,0,0,0


In [8]:
conn = get_db_connection()
pd.read_sql_query("select * from evaluations where session_id=5", conn)

,id,session_id,prompt,meme_template_id,gemini_json,imgflip_url,is_humor,is_creative,is_shareable,comments,error,timestamp
0,88,5,"I am trying to make a funny meme, but my sense...",None,"{""error"": ""Failed after 3 attempts. Last error...",None,0,0,0,Variation 1 selected. Error,Failed after 3 attempts. Last error: Invalid J...,2025-04-28T15:08:08.035139
1,89,5,"I am trying to make a funny meme using AI, but...",None,"{""error"": ""Failed after 3 attempts. Last error...",None,0,0,0,Variation 1 selected. Error,Failed after 3 attempts. Last error: Invalid J...,2025-04-28T15:08:56.148909
2,90,5,Pharmacy overlord,None,"{""error"": ""Failed after 3 attempts. Last error...",None,0,0,0,Variation 1 selected. Error,Failed after 3 attempts. Last error: Invalid J...,2025-04-28T15:09:14.853232


In [9]:
# issue in the logs - we don't see the raw text output
errors = pd.read_sql_query("select * from evaluations where session_id=5", conn)['gemini_json'].to_list()
for error in errors:
    print(error)
    print('-'*100)


{"error": "Failed after 3 attempts. Last error: Invalid JSON in model response (attempt 3/3): Expecting value: line 1 column 1 (char 0)"}
----------------------------------------------------------------------------------------------------
{"error": "Failed after 3 attempts. Last error: Invalid JSON in model response (attempt 3/3): Expecting value: line 1 column 1 (char 0)"}
----------------------------------------------------------------------------------------------------
{"error": "Failed after 3 attempts. Last error: Invalid JSON in model response (attempt 3/3): Expecting value: line 1 column 1 (char 0)"}
----------------------------------------------------------------------------------------------------


In [10]:
import os
import json
import google.generativeai as genai
from dotenv import load_dotenv

# Import the functions we need from the existing code
from utils.gemini import initialize_model, construct_meme_prompt, clean_response

# Load environment variables
load_dotenv()

# Load system prompt
with open('data/system_prompt.txt', 'r') as f:
    system_prompt = f.read()

# Load meme context
with open('data/memes20250304.json', 'r') as f:
    meme_context = json.load(f)

# Initialize model
MODEL_NAME = 'gemini-2.0-flash-lite'
model = initialize_model(os.getenv('GEMINI_API_KEY'), MODEL_NAME, system_prompt)

def test_prompt(prompt_text, show_raw=True, max_attempts=1):
    """
    Test a prompt and show both raw and cleaned outputs.
    
    Args:
        prompt_text: The prompt to test
        show_raw: Whether to show the raw model output
        max_attempts: Number of attempts to try
        
    Returns:
        Dictionary containing raw output and cleaned JSON
    """
    config = {
        'max_output_tokens': 1000,
        'temperature': 0.1,
    }
    
    meme_prompt = construct_meme_prompt(prompt_text, meme_context)
    
    results = []
    for attempt in range(max_attempts):
        try:
            # Generate completion
            response = model.generate_content(
                meme_prompt,
                generation_config=genai.GenerationConfig(**config)
            )
            
            raw_output = response.text
            cleaned = clean_response(raw_output)
            
            try:
                json_output = json.loads(cleaned)
            except json.JSONDecodeError:
                json_output = None
                
            result = {
                'attempt': attempt + 1,
                'raw_output': raw_output,
                'cleaned_output': cleaned,
                'parsed_json': json_output,
                'is_valid_json': json_output is not None
            }
            
            results.append(result)
            
        except Exception as e:
            results.append({
                'attempt': attempt + 1,
                'error': str(e)
            })
    
    return results

# Example usage:
test_prompt_text = "Create a meme about debugging code"
results = test_prompt(test_prompt_text)

# Display results in a more readable format
for result in results:
    print("=== Attempt", result.get('attempt'), "===")
    if 'error' in result:
        print("Error:", result['error'])
        continue
        
    print("\nRAW OUTPUT:")
    print(result['raw_output'])
    print("\nCLEANED OUTPUT:")
    print(result['cleaned_output'])
    print("\nPARSED JSON:")
    print(json.dumps(result['parsed_json'], indent=2) if result['parsed_json'] else "Failed to parse JSON")
    print("\nIs valid JSON:", result['is_valid_json'])
    print("\n" + "="*50 + "\n")

=== Attempt 1 ===

RAW OUTPUT:
```json
{
    "template_id": "89370399",
    "text0": "Can't fix the bug",
    "text1": "If you don't understand the code"
}
```

CLEANED OUTPUT:
json
{
    "template_id": "89370399",
    "text0": "Can"t fix the bug",
    "text1": "If you don"t understand the code"
}

PARSED JSON:
Failed to parse JSON

Is valid JSON: False


